# Qwen3.8-Flash-Next + LLKVApprox v2 (late-layer KV/state approximation) on 2x CMP 170HX (SM80), packed int4 experts + HF eager bf16

| Metric | Value |
|---|---|
| Mechanism validated (oracle fills, single-pass capture) | bit-exact: FA prior K bit-identical (0.0), GDN grid-boundary replay exact-by-construction; residual = MoE routing flips (see gate note) |
| Prefill speed ceiling (oracle, trained-fill cost excluded) | **2.19-2.27x** @1,024-6,603 tok (e.g. 6,603 tok: 119.1 vs 53.4 tok/s) |
| Ridge linear ceiling (24.5k teacher tok) | GDN qkv **0.949**, b 0.981, a 0.976 — FA k 0.62 / v 0.71 / index_k 0.63 (per-layer own-weights + MLP heads required for FA) |
| Stage-1 distillation | **completed**: 2,000 steps / 0.88 h cached-target (zero teacher kernels), final FA k 0.953 / v 0.974 / index_k 0.960, GDN k 0.982 / v 0.991; holdout matches train |
| Trained student greedy match (suffix 1, 48-tok chains) | **69-83% per prompt** (v1 27B trained suffix-1: 7.4%) |

Reference mechanism: [LLKVApprox demo](https://kishida.github.io/webdemos/llkvapprox/) (Qwen3-8B, dense) and
[the author's write-up](https://nowokay.hatenablog.com/entry/2026/09/11/120001). Projector:
[PixelML/Qwen3.8-Flash-Next-KVA-Projector](https://huggingface.co/PixelML/Qwen3.8-Flash-Next-KVA-Projector).


# Qwen3.8-Flash-Next + LLKVApprox v2 — 2x CMP 170HX, packed int4 experts + HF eager

Port of the v1 LLKVApprox port (Qwen3.8-27B, dense, 1 card) to **Qwen3.8-Flash-Next**
(`qwen4_exp`): 48 layers = 36 gated-delta-net + 12 full attention, **512-expert MoE in every
layer** (top-10), 4-stream **hyper-connections**, a **lightning indexer** per FA layer, PLE
n-gram tables, and an MTP head (unused here). v1's dense model had none of these; each one
changed the port:

- **MoE everywhere**: bf16-resident experts are impossible (121B expert params = 242G), so the
  engine keeps experts int4-packed on GPU (~35G/card) and dequantizes routed experts per
  forward; PLE n-gram tables (~102G) stay CPU-mmap'd. Layer split 24 = CED boundary = card
  boundary (encoder on GPU0, decoder on GPU1).
- **Hyper-connections**: the residual stream is 4x2560; the projector's boundary state is
  10240-dim and the own-weights init becomes the block-average of the layer's in_proj per stream.
- **Lightning indexer (issue open question, resolved: yes)**: every FA layer caches raw per-token
  indexer keys and the block selection consumes them — approximated FA layers need **K, V, and
  index_k** filled (a third projector target v1 did not have).
- **Chunked delta-rule non-associativity**: splitting 256 as 255+1 perturbs the GDN state by
  ~10% rel; the oracle injects at the last 64-chunk boundary and replays the tail through the
  teacher's own chunked kernel (bit-exact by construction).
- **MoE routing flips set the oracle noise floor**: bf16-ulp GEMM differences between the CED
  suffix path (M=1) and the full prefill (M=T) flip near-tie router decisions and amplify ~100x
  through the stack. The model is otherwise fully deterministic (shape-noise floor = 0.0), so
  v1's dense-model 0.013 gate is unreachable on this architecture; the gate is recalibrated to
  bit-exact fills + bounded drift + argmax agreement (see the oracle cell).

Evidence source: committed `receipts/` JSON recorded 2026-09-14/15. Every number below is read
from those receipts; nothing is re-measured by this notebook (LIVE = False).


In [1]:
# --- Status cell -------------------------------------------------------
import os

EXPERIMENT = "2026-09-15-qwen3.8-flash-next-llkvapprox-2card"
RESULTS_DIR = os.path.join("..", "receipts", EXPERIMENT)
LIVE = False

print(f"experiment   : {EXPERIMENT}")
print(f"results_dir  : {RESULTS_DIR}")
print(f"LIVE         : {LIVE}")


experiment   : 2026-09-15-qwen3.8-flash-next-llkvapprox-2card
results_dir  : ../receipts/2026-09-15-qwen3.8-flash-next-llkvapprox-2card
LIVE         : False


In [2]:
# --- Helpers ------------------------------------------------------------
import json, os

def load_receipt(name, results_dir=RESULTS_DIR):
    with open(os.path.join(results_dir, name)) as f:
        return json.load(f)

def render_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    from IPython.display import display, Markdown
    display(Markdown("\n".join(lines)))


In [3]:
# --- Feasibility: packed-resident load, 2x 64GB cards -------------------
smoke = load_receipt("load-smoke.json")
render_table(
    ["item", "value"],
    [
        ["GPU0 / GPU1 resident", f"{smoke['gpu0_gb']} / {smoke['gpu1_gb']} GiB"],
        ["layer split", "0..23 cuda:0 (encoder) | 24..47 cuda:1 (decoder)"],
        ["load time", f"{smoke['load_seconds']} s"],
        ["expert dequant matches compressed-tensors reference", smoke["dequant_matches_ct"]],
        ["512-token full prefill", f"{smoke['prefill_512_s']} s, finite logits: {smoke['prefill_512_finite']}"],
    ],
)


| item | value |
|---|---|
| GPU0 / GPU1 resident | 37.6 / 37.6 GiB |
| layer split | 0..23 cuda:0 (encoder) | 24..47 cuda:1 (decoder) |
| load time | 341.2 s |
| expert dequant matches compressed-tensors reference | True |
| 512-token full prefill | 67.81 s, finite logits: True |

In [4]:
# --- Speed ceiling: full vs oracle CED prefill --------------------------
bench = load_receipt("bench-prefill.json")
rows = [[r["tokens"], f"{r['full_s']} / {r['full_tok_s']}", f"{r['oracle_s']} / {r['oracle_tok_s']}",
         f"**{r['speedup']}x**"] for r in bench["rows"]]
render_table(["tokens", "full s / tok/s", "oracle s / tok/s", "speedup"], rows)
print("timed region = encoder + grid-replay fills + suffix; fills come from the pass bundle\n"
      "(in deployment: the trained projector). Both arms share every eager path.")


| tokens | full s / tok/s | oracle s / tok/s | speedup |
|---|---|---|---|
| 512 | 72.421 / 7.1 | 58.833 / 8.7 | **1.23x** |
| 1024 | 88.809 / 11.5 | 39.15 / 26.2 | **2.27x** |
| 2048 | 98.753 / 20.7 | 45.122 / 45.4 | **2.19x** |
| 6603 | 123.606 / 53.4 | 55.44 / 119.1 | **2.23x** |

timed region = encoder + grid-replay fills + suffix; fills come from the pass bundle
(in deployment: the trained projector). Both arms share every eager path.


In [5]:
# --- Oracle gate (MoE-aware) --------------------------------------------
o = load_receipt("oracle-test.json")
print("last position:", o["last_position"])
print("drift:", o["teacher_forced_drift"])
print("gate note:", o["gate_note"])


last position: {'mean_abs': 0.6796875, 'max_abs': 6.84375, 'median_rel': 0.26171875, 'argmax_match': True, 'top5_overlap': 3}
drift: {'steps': 32, 'mean_abs_dlogit_first8': 0.87451, 'mean_abs_dlogit_last8': 0.36877, 'max_mean_abs': 1.30469, 'argmax_match': '28/32', 'flat': False}
gate note: bit-exact state fills (FA priors 0.0, GDN grid-boundary replay); residual = GEMM-shape ulp noise amplified by MoE routing flips (architectural, irreducible in CED on this model)


In [6]:
# --- Why the dense-model gate is unreachable: chunked-kernel + MoE flips -
probe = load_receipt("gdn-kernel-probe.json")
print("chunked delta-rule: single-call chunk(0..255) vs stored state:",
      probe["B_chunked256_vs_true"])
print("split 255 + chunk(1):", probe["A_fused_from_S254_vs_true"])
floor = load_receipt("shape-noise-floor.json")
print("shape-noise floor (same rows, longer GEMM batch):", floor["shape_noise_last_position"],
      floor["shape_noise_drift"])


chunked delta-rule: single-call chunk(0..255) vs stored state: {'mean_abs': 0.0, 'max_abs': 0.0, 'rel_fro': 0.0}
split 255 + chunk(1): {'mean_abs': 7.248293695738539e-05, 'max_abs': 0.11569756269454956, 'rel_fro': 0.0996759831905365}
shape-noise floor (same rows, longer GEMM batch): {'mean_abs': 0.0, 'max_abs': 0.0, 'median_rel': 0.0, 'argmax_match': True, 'top5_overlap': 5} {'argmax_match': '32/32', 'mean_abs_first8': 0.0, 'mean_abs_last8': 0.0, 'max': 0.0}


In [7]:
# --- Ridge linear ceiling (closed form, 24.5k teacher tokens) ------------
ridge = load_receipt("ridge-ceiling.json")
rows = [[name, v["dims"], v["heldout_cosine"], f"lambda x{v['best_lambda_mul']}"]
        for name, v in ridge["ridge"].items()]
render_table(["target", "dims", "held-out cosine", "lambda"], rows)
print("FA targets need per-layer own-weights + MLP heads; GDN is linear-predictable.")


| target | dims | held-out cosine | lambda |
|---|---|---|---|
| fa_k | 512 | 0.6216 | lambda x0.001 |
| fa_v | 512 | 0.7094 | lambda x0.001 |
| fa_ik | 128 | 0.627 | lambda x0.001 |
| gdn_qkv | 10240 | 0.9485 | lambda x0.001 |
| gdn_b | 48 | 0.9812 | lambda x0.001 |
| gdn_a | 48 | 0.9762 | lambda x0.001 |

FA targets need per-layer own-weights + MLP heads; GDN is linear-predictable.


In [8]:
# --- Stage-1 cached training (zero teacher kernels) ----------------------
tr = load_receipt("stage1_cached_v2.json")
print("args:", {k: tr["args"][k] for k in ("steps", "lr", "tag")})
print("history tail:")
for rec in tr["history_tail"][-3:]:
    print(" ", rec)
print("holdout eval:")
for e in tr["holdout_eval"]:
    print(" ", e)
print("wall hours:", tr["wall_hours"])


args: {'steps': 2000, 'lr': 0.001, 'tag': 'stage1_cached_v2'}
history tail:
  {'step': 1950, 'loss': 4.2234, 's_per_step': 1.57, 'fa_cos_k': 0.9292, 'fa_cos_v': 0.9604, 'fa_cos_ik': 0.9408, 'gdn_cos_k': 0.9644, 'gdn_cos_v': 0.9842}
  {'step': 1975, 'loss': 2.6435, 's_per_step': 1.571, 'fa_cos_k': 0.9493, 'fa_cos_v': 0.9736, 'fa_cos_ik': 0.9577, 'gdn_cos_k': 0.9812, 'gdn_cos_v': 0.9904}
  {'step': 1999, 'loss': 2.512, 's_per_step': 1.574, 'fa_cos_k': 0.9525, 'fa_cos_v': 0.9741, 'fa_cos_ik': 0.9603, 'gdn_cos_k': 0.982, 'gdn_cos_v': 0.9909}
holdout eval:
  {'seq': 'seq_013.pt', 'loss': 2.6197, 'fa_cos_k': 0.9503, 'gdn_cos_k': 0.9811}
  {'seq': 'seq_014.pt', 'loss': 2.512, 'fa_cos_k': 0.9525, 'gdn_cos_k': 0.982}
  {'seq': 'seq_015.pt', 'loss': 3.0232, 'fa_cos_k': 0.9452, 'gdn_cos_k': 0.9779}
wall hours: 0.88


In [9]:
# --- Trained student quality (greedy match vs baseline, diagnostic) ------
q = load_receipt("quality-trained.json")
print("summary:", json.dumps(q["summary"], indent=1))
print(q["note"])


summary: {
 "suffix1": {
  "greedy_match": "255/336 = 75.9%",
  "mean_ttft_s": 12.12
 },
 "suffix256": {
  "greedy_match": "336/336 = 100.0%",
  "mean_ttft_s": 26.55
 }
}
greedy token match vs full baseline is a DIAGNOSTIC, not task accuracy; short prompts (T<=256) take the exact path at suffix=256


# ### Reproduce
#
| Pin | Value |
|---|---|
| Model | Qwen3.8-Flash-Next (qwen4_exp), packaged from `Qwen3.8-Flash-Next-AWQ-INT4` weights (176G, compressed-tensors pack-quantized) |
| Hardware | 2x CMP 170HX 64GB @250W (SM80), host-offloaded PLE |
| Runtime | torch 2.14.0+cu130, transformers 5.18.0.dev0 (`qwen4_exp`), fla 0.6.0, compressed-tensors 0.18.0 (reference conv fallback; `causal_conv1d` not buildable on this host) |
| Projector | [PixelML/Qwen3.8-Flash-Next-KVA-Projector](https://huggingface.co/PixelML/Qwen3.8-Flash-Next-KVA-Projector) (385MB bf16) |
| Engine | issue #139 progress comments + `~/WIP/llkvapprox-flash-next/` (src + scripts + receipts) |
| Date | receipts recorded 2026-09-14/15 |
#
# Every number above reads the committed receipts; this notebook re-measures nothing.


# ### Appendix: layer-input divergence profile (oracle vs baseline, single-pass capture)
#
# From `oracle-debug.json`: layer 24 input bit-exact (0.0), FA prior K bit-exact (0.0);
# divergence enters at the GDN row-255 decode step (bf16-ulp GEMM-shape noise, M=1 vs M=T),
# then jumps at the first near-tie router flip (layer-input rel_fro 0.0066 -> 0.72 at layer 28)
# and stays bounded thereafter. Full per-layer profile in the receipt.
